# Institution resource-list cleanup

Post-processes the output of the `accredited-institution-resources` skill. Two independent
operations, each reading an existing xlsx and writing a **new** file next to it — neither
operation ever overwrites its input. See `../SKILL.md` for the full rationale behind each.

Both operations expect at least these columns: `unitid, institution_name, homepage_url, city,
state, accreditor, pct_biomedical, has_biomedical_program_data, base_domain, resource_list_url,
resource_list_notes`.

## Operation 1: tighten the religious-institution and arts-school filters

Adds signals beyond what `accredited-institution-resources`'s Filters 4/5 already check
(arts/music program-percentage + narrow name pattern; faith-related accreditor name +
theology-program share), for cases spotted during Step 5 that slipped past both:

**Religious institutions** (Filter 5):
- **Name-pattern match** — catches institutions whose `accreditor` field in the Scorecard API only
  lists a regional accreditor even though they're also accredited by a faith-related one in reality
  (e.g. Southeastern Baptist Theological Seminary is ATS-accredited, but the API only returns
  SACSCOC for it).
- **Manual override list** (`../reference/manual_religious_institution_overrides.csv`) — a small,
  curated list of institutions with no religious keyword in their name at all (Regent University,
  Calvin University, Spertus College, Columbia International University, Hebrew College, Thomas
  Aquinas College - New England), verified religious by other means.

**Arts/design schools** (Filter 4):
- **Name-pattern match** — a couple of exact-phrase additions to catch schools the original
  narrow arts regex missed: `college of the arts` (California College of the Arts), `college of
  fine arts` (Vermont College of Fine Arts), bare `conservatory` (American Film Institute
  Conservatory), and `visual arts` (Institute for Doctoral Studies in the Visual Arts). Each was
  checked against the full institution list before adding to confirm zero false positives — e.g. a
  broader `college of arts?\b` was tried first and rejected because it matched "Paul Smiths College
  of Arts and Science," a comprehensive college that's not an art school at all; the exact phrases
  above avoid that.
- **Manual override list** (`../reference/manual_arts_institution_overrides.csv`) — institutions
  with no art/design keyword in their name at all (Pratt Institute, Cooper Union — whose "Art"
  appears only inside a specific historical phrase too fragile to safely generalize into a pattern)
  or a generic-sounding name (College for Creative Studies).

Both are kept as explicit lists rather than ever-widening the regexes, to avoid false-positiving on
secular/non-arts institutions with similar-sounding names.

In [1]:
import pathlib
import re

import pandas as pd

# Point this at whatever spreadsheet needs the tightened filters applied.
INPUT_PATH = pathlib.Path("../../output/2026-07-31_newly_added_institutions_step5.xlsx")

df = pd.read_excel(INPUT_PATH)

RELIGIOUS_NAME_PATTERN = re.compile(
    r"\bseminary\b|\byeshiva\b|\btalmudic\b|\brabbinical\b|\btorah\b|"
    r"jewish institute of religion|bible institute|bible college|biblical institute|"
    r"biblical seminary|school of theology|theological school|theological seminary|"
    r"divinity school|school of divinity|\btheological\b",
    re.IGNORECASE,
)

# Deliberately exact phrases, not a general "college of arts?" pattern -- that broader version was
# tried first and rejected because it matched "Paul Smiths College of Arts and Science" (a
# comprehensive college, not an art school).
ARTS_NAME_PATTERN = re.compile(
    r"college of the arts|college of fine arts|\bconservatory\b|visual arts\b",
    re.IGNORECASE,
)

religious_overrides = pd.read_csv("../reference/manual_religious_institution_overrides.csv")
religious_override_unitids = set(religious_overrides["unitid"])

arts_overrides = pd.read_csv("../reference/manual_arts_institution_overrides.csv")
arts_override_unitids = set(arts_overrides["unitid"])

religious_name_match = df["institution_name"].str.contains(RELIGIOUS_NAME_PATTERN, na=False)
religious_override_match = df["unitid"].isin(religious_override_unitids)
arts_name_match = df["institution_name"].str.contains(ARTS_NAME_PATTERN, na=False)
arts_override_match = df["unitid"].isin(arts_override_unitids)

remove_mask = religious_name_match | religious_override_match | arts_name_match | arts_override_match

removed = df[remove_mask]
print(f"{len(removed)} rows matched across {removed['base_domain'].nunique()} domains:")
print(f"  religious: {int(religious_name_match.sum())} by name pattern, "
      f"{int(religious_override_match.sum())} by manual override")
print(f"  arts:      {int(arts_name_match.sum())} by name pattern, "
      f"{int(arts_override_match.sum())} by manual override")
print()
for idx, row in removed.iterrows():
    if religious_name_match[idx]:
        reason = "religious name pattern"
    elif religious_override_match[idx]:
        reason = "religious manual override"
    elif arts_name_match[idx]:
        reason = "arts name pattern"
    else:
        reason = "arts manual override"
    print(f"  {row['unitid']} | {row['institution_name']} | {row['base_domain']} | {reason}")

filtered = df[~remove_mask].copy()

FILTERED_OUTPUT_PATH = INPUT_PATH.with_name(INPUT_PATH.stem + "_filtered" + INPUT_PATH.suffix)
filtered.to_excel(FILTERED_OUTPUT_PATH, index=False)

print()
print(f"Input:  {len(df)} rows ({INPUT_PATH})")
print(f"Output: {len(filtered)} rows ({FILTERED_OUTPUT_PATH})")

6 rows matched across 6 domains:
  religious: 0 by name pattern, 2 by manual override
  arts:      4 by name pattern, 0 by manual override

  108870 | American Film Institute Conservatory | afi.com | arts name pattern
  110370 | California College of the Arts | cca.edu | arts name pattern
  166045 | Hebrew College | hebrewcollege.edu | religious manual override
  462044 | Institute for Doctoral Studies in the Visual Arts | idsva.edu | arts name pattern
  12429201 | Thomas Aquinas College - New England | thomasaquinas.edu | religious manual override
  455992 | Vermont College of Fine Arts | vcfa.edu | arts name pattern

Input:  291 rows (..\..\output\2026-07-31_newly_added_institutions_step5.xlsx)
Output: 285 rows (..\..\output\2026-07-31_newly_added_institutions_step5_filtered.xlsx)


## Operation 2: de-duplicate by resource URL

Many institutions share the exact same `resource_list_url` — not just branch campuses of the same
system (already grouped by `base_domain` upstream), but also different `base_domain`s that
independently fell back to the same shared page (e.g. several Penn State branch-campus domains all
fall back to the same main Penn State Libraries page). De-duplicating by `base_domain` alone doesn't
collapse these; de-duplicating by `resource_list_url` does.

Output keeps the same columns, one row per unique `resource_list_url`. Where multiple institutions
share a URL, their identifying fields become a `; `-joined list of every institution using that URL
— nothing is silently dropped. `pct_biomedical` takes the max across the group (preserving the
upstream sort/priority behavior) and `has_biomedical_program_data` is true if any member has it.

`INPUT_PATH_2` defaults to the same base spreadsheet as Operation 1 (independent of it) — point it
at `FILTERED_OUTPUT_PATH` instead if you want to run the two operations chained.

In [2]:
import pathlib

import pandas as pd

INPUT_PATH_2 = FILTERED_OUTPUT_PATH  # chained: dedupe the just-tightened output from Operation 1

COLS = [
    "unitid", "institution_name", "homepage_url", "city", "state", "accreditor",
    "pct_biomedical", "has_biomedical_program_data", "base_domain",
    "resource_list_url", "resource_list_notes",
]

df2 = pd.read_excel(INPUT_PATH_2)


def join_unique(series):
    seen = []
    for v in series:
        s = "" if pd.isna(v) else str(v)
        if s not in seen:
            seen.append(s)
    return "; ".join(seen)


grouped = df2.groupby("resource_list_url", sort=False).agg(
    unitid=("unitid", join_unique),
    institution_name=("institution_name", join_unique),
    homepage_url=("homepage_url", join_unique),
    city=("city", join_unique),
    state=("state", join_unique),
    accreditor=("accreditor", join_unique),
    pct_biomedical=("pct_biomedical", "max"),
    has_biomedical_program_data=("has_biomedical_program_data", "any"),
    base_domain=("base_domain", join_unique),
    resource_list_notes=("resource_list_notes", join_unique),
).reset_index()

grouped = grouped[COLS]
grouped = grouped.sort_values(["pct_biomedical", "institution_name"], ascending=[False, True]).reset_index(drop=True)

DEDUPED_OUTPUT_PATH = INPUT_PATH_2.with_name(INPUT_PATH_2.stem + "_unique_resource_urls" + INPUT_PATH_2.suffix)
grouped.to_excel(DEDUPED_OUTPUT_PATH, index=False)

n_merged_groups = int((df2.groupby("resource_list_url").size() > 1).sum())
print(f"Input:  {len(df2)} rows, {df2['resource_list_url'].nunique()} unique resource_list_url values ({INPUT_PATH_2})")
print(f"Output: {len(grouped)} rows, one per unique resource_list_url ({DEDUPED_OUTPUT_PATH})")
print(f"{n_merged_groups} groups had 2+ institutions merged into one row")

Input:  285 rows, 254 unique resource_list_url values (..\..\output\2026-07-31_newly_added_institutions_step5_filtered.xlsx)
Output: 254 rows, one per unique resource_list_url (..\..\output\2026-07-31_newly_added_institutions_step5_filtered_unique_resource_urls.xlsx)
19 groups had 2+ institutions merged into one row
